In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('ticks',
              rc={'axes.facecolor': (0, 0, 0, 0)})
sns.set_context('talk')

from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'
rcParams['figure.dpi'] = 150

In [2]:
import os
import itertools
import numpy as np
import pandas as pd

In [3]:
d = {
    'M0': 'PL+PM clone 1', 
    'M1': 'PL+PM clone 2', 
    'M2': 'PL+LM clone 1',
    'M3': 'PL+LM clone 2',
    'M4': 'PM+LM clone 1',
    'M5': 'PM+LM clone 2',
    'M6': 'TriComm clone 1',
    'M7': 'TriComm clone 2'
}
hue_order = list(d.values())

In [4]:
EMISSIONS = ['FP1_emit1', 'FP2_emit1', 'FP3_emit1']

def find_turbidostat(v):
    x = v[v['pump_1_rate'] > 0].sort_values('time')[['time',
                                                     'pump_1_rate']].reset_index(drop=True)
    x['delta'] = x['time'].diff()
    
    y = x[x['delta'] < 2].sort_values('time').reset_index(drop=True)
    y['delta2'] = y['time'].diff()

    if y.shape[0] == 0:
        return None
    
    times = []
    start = y['time'].values[0]
    end = None
    for time in y[y['delta2'] > 2]['time'].values:
        if start is None:
            start = time
            end = None
            continue
        end = time
        times.append((start, end))
        start = None
    if end is None:
        times.append((start, start + 2))

    return times

def parse_chibio(rep):
    res = []
    for f in os.listdir('chibio'):
        if f.endswith('_data.csv'):
            # for plate in ['AMP', 'KAN']:
            m = read_chibio(f'chibio/{f}', emissions=EMISSIONS, overlay=True)
            name = f.split('_')[3]
            if name not in d:
                continue
            m['vial'] = name
            m['name'] = d[name]
            m['short'] = d[name].replace('TriComm ', '')
            m['plasmid'] = d[name].split()[1]
            m['drug'] = 0 if 'MIC' not in d[name] else int(d[name].split('x')[0][-2:])
            m['replicate'] = rep
            # m['plate'] = plate
            res.append(m)
    m = pd.concat(res).reset_index(drop=True)

    m = m.sort_values(['vial', 'exp_time']).reset_index(drop=True)
    
    m['smooth_masked_FP1_emit1'] = m['smooth_FP1_emit1'].values
    m['smooth_masked_FP2_emit1'] = m['smooth_FP2_emit1'].values
    m['smooth_norm_masked_FP1_emit1'] = m['smooth_norm_FP1_emit1'].values
    m['smooth_norm_masked_FP2_emit1'] = m['smooth_norm_FP2_emit1'].values
    for em1, em2 in itertools.combinations(EMISSIONS, 2):
        m[f'masked_{em1}_{em2}_ratio'] = m[f'{em1}_{em2}_ratio']
        m[f'smooth_masked_{em1}_{em2}_ratio'] = m[f'smooth_{em1}_{em2}_ratio']
        m[f'smooth_norm_masked_{em1}_{em2}_ratio'] = m[f'smooth_norm_{em1}_{em2}_ratio']
    for vial in d:
        times = find_turbidostat(m[m['vial'] == vial])
        if times is None:
            continue
        for start, end in times:
            idx = m[(m['vial'] == vial) &
                    (m['time'] >= start - 0.5) &
                    (m['time'] <= end + 0.5)].index
            m.loc[idx, 'smooth_masked_FP1_emit1'] = np.nan
            m.loc[idx, 'smooth_masked_FP2_emit1'] = np.nan
            m.loc[idx, 'smooth_norm_masked_FP1_emit1'] = np.nan
            m.loc[idx, 'smooth_norm_masked_FP2_emit1'] = np.nan
            for em1, em2 in itertools.combinations(EMISSIONS, 2):
                m.loc[idx, f'masked_{em1}_{em2}_ratio'] = np.nan
                m.loc[idx, f'smooth_masked_{em1}_{em2}_ratio'] = np.nan
                m.loc[idx, f'smooth_norm_masked_{em1}_{em2}_ratio'] = np.nan
    
    cols = ['vial', 'replicate', 'name', 'short', 'plasmid', 'drug',
            'time', 'smooth_od', 'ln(od)', 'smooth_FP1_emit1', 'smooth_norm_FP1_emit1',
            'smooth_FP2_emit1', 'smooth_norm_FP2_emit1', 'smooth_FP3_emit1',
            'smooth_norm_FP3_emit1', 
            'smooth_masked_FP1_emit1', 'smooth_masked_FP2_emit1',
            'smooth_norm_masked_FP1_emit1', 'smooth_norm_masked_FP2_emit1',
            'FP1_emit1_FP2_emit1_ratio', 'smooth_FP1_emit1_FP2_emit1_ratio',
            'masked_FP1_emit1_FP2_emit1_ratio', 'smooth_masked_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_masked_FP1_emit1_FP2_emit1_ratio',
            'smooth_norm_FP1_emit1_FP2_emit1_proportion',
            'smooth_norm_FP2_emit1_FP1_emit1_proportion',
            'exp_time', 'od_measured', 'od_setpoint', 'od_zero_setpoint',
            'thermostat_setpoint', 'heating_rate', 'internal_air_temp',
            'external_air_temp', 'media_temp', 'opt_gen_act_int', 'pump_1_rate',
            'pump_2_rate', 'pump_3_rate', 'pump_4_rate', 'media_vol',
            'stirring_rate', 'LED_395nm_setpoint', 'LED_457nm_setpoint',
            'LED_500nm_setpoint', 'LED_523nm_setpoint', 'LED_595nm_setpoint',
            'LED_623nm_setpoint', 'LED_6500K_setpoint', 'laser_setpoint',
            'LED_UV_int', 'FP1_base', 'FP1_emit1', 'FP1_emit2', 'FP2_base',
            'FP2_emit1', 'FP2_emit2', 'FP3_base', 'FP3_emit1', 'FP3_emit2',
            'custom_prog_param1', 'custom_prog_param2', 'custom_prog_param3',
            'custom_prog_status', 'zigzag_target', 'growth_rate']
    m = m[cols].sort_values(['vial', 'time']).reset_index(drop=True)

    return m

m = parse_chibio('')

m['replicate'] = [f'rep{int(x[-1])}' for x in m['name'].values]
m['name'] = [x.split()[0] for x in m['name'].values]

m.to_csv('chibio.tsv.gz', sep='\t', index=False)

NameError: name 'read_chibio' is not defined